# Data-Driven Real Estate Investment Decision Model under Uncertainty

**Author:** Quantitative Research Portfolio Project  \n**Objective:** Build a rigorous end-to-end framework for real estate investment decisioning that combines predictive ML, stochastic risk simulation, and corporate-finance valuation analytics.

---

## Project Structure
1. Problem framing and assumptions
2. Data generation and feature engineering
3. Supervised learning model comparison
4. Financial cash-flow model (levered investment)
5. Monte Carlo simulation (10,000+ scenarios)
6. Sensitivity and scenario analysis
7. Conclusions and limitations


## 1) Modeling assumptions and design rationale

Because high-quality transaction-level panel data is often proprietary, we construct a **synthetic but economically structured** dataset. The generator encodes realistic causal relationships between:
- hedonic property features (size, bedrooms, age),
- neighborhood quality (school, transit, crime),
- macro regimes (mortgage rates, unemployment, inflation, GDP growth), and
- cyclical + trend dynamics in housing values.

This setup preserves quantitative rigor while ensuring full reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.real_estate_investment_model import (
    InvestmentAssumptions,
    create_model_matrix,
    evaluate_models,
    generate_synthetic_real_estate_data,
    monte_carlo_investment,
    npv,
    irr,
    run_sensitivity_grid,
    build_cashflows,
)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
Path('../results/figures').mkdir(parents=True, exist_ok=True)

## 2) Synthetic data generation and feature engineering

In [ ]:
df = generate_synthetic_real_estate_data(n_properties=3000, years=10, seed=42)
X, y, features = create_model_matrix(df)

print('Dataset shape:', df.shape)
df.head()

In [ ]:
summary = df[['sale_price', 'monthly_rent', 'mortgage_rate', 'unemployment_rate', 'inflation_rate', 'gdp_growth']].describe().T
summary

## 3) Machine-learning model comparison

We benchmark three models:
- Linear Regression (transparent baseline),
- Random Forest (non-linear ensemble),
- Gradient Boosting (strong tabular predictive model).

Performance is evaluated by out-of-sample **R²** and **RMSE** on log-price predictions.

In [ ]:
metrics, fitted_models, splits = evaluate_models(X, y, random_state=42)
metrics

In [ ]:
best_model_name = metrics.iloc[0]['model']
best_model = fitted_models[best_model_name]
print('Selected model:', best_model_name)

In [ ]:
# Feature importance (tree-based if available)
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False).head(12)
else:
    fi = pd.Series(np.abs(best_model.coef_), index=features).sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=fi.values, y=fi.index, ax=ax, color='#2A6F97')
ax.set_title(f'Top Feature Contributions ({best_model_name})')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
fig.tight_layout()
fig.savefig('../results/figures/feature_importance.png', bbox_inches='tight')
plt.show()

## 4) Financial model: levered acquisition cash flows

We model a 7-year hold with leverage, amortizing debt, operating cash flows, and terminal exit proceeds net of sale costs.

**Core metrics:**
- Net Present Value (NPV),
- Internal Rate of Return (IRR),
- Payback period.

In [ ]:
assump = InvestmentAssumptions()
base_price_growth = np.full(assump.holding_period_years, assump.annual_price_growth_mu)
base_rent_growth = np.full(assump.holding_period_years, assump.rent_growth_mu)

base_cf = build_cashflows(assump, base_price_growth, base_rent_growth)
cf = base_cf['cashflows']

base_results = {
    'NPV': npv(assump.discount_rate, cf),
    'IRR': irr(cf),
    'Payback (years)': base_cf['payback_period'],
    'Terminal Property Value': base_cf['property_value_terminal'],
    'Net Sale Proceeds': base_cf['sale_proceeds'],
}
pd.Series(base_results).to_frame('Base Case')

## 5) Monte Carlo risk simulation (10,000 scenarios)

Uncertainty is introduced in annual price growth and rent growth with user-defined means and volatilities.
For each path we compute full levered cash flows, IRR, and NPV.

In [ ]:
mc = monte_carlo_investment(assump, n_sims=12000, seed=7)
mc.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(mc['irr'].dropna(), bins=60, kde=True, ax=axes[0], color='#0FA3B1')
axes[0].axvline(mc['irr'].median(), color='black', linestyle='--', linewidth=1.5, label='Median')
axes[0].set_title('Distribution of Simulated IRR')
axes[0].set_xlabel('IRR')
axes[0].legend()

sns.histplot(mc['npv'], bins=60, kde=True, ax=axes[1], color='#F4A261')
axes[1].axvline(mc['npv'].median(), color='black', linestyle='--', linewidth=1.5, label='Median')
axes[1].set_title('Distribution of Simulated NPV')
axes[1].set_xlabel('NPV (USD)')
axes[1].legend()

fig.tight_layout()
fig.savefig('../results/figures/return_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
prob_negative_npv = (mc['npv'] < 0).mean()
prob_irr_below_hurdle = (mc['irr'] < 0.12).mean()

risk_summary = pd.DataFrame({
    'Metric': ['P(NPV < 0)', 'P(IRR < 12%)', 'VaR 5% (NPV)', 'CVaR 5% (NPV)'],
    'Value': [
        prob_negative_npv,
        prob_irr_below_hurdle,
        mc['npv'].quantile(0.05),
        mc.loc[mc['npv'] <= mc['npv'].quantile(0.05), 'npv'].mean(),
    ]
})
risk_summary

## 6) Sensitivity and scenario analysis

In [ ]:
sens = run_sensitivity_grid(assump)
sens.head()

In [ ]:
pivot_npv = sens.pivot_table(index='interest_rate', columns='purchase_price', values='npv', aggfunc='mean')
pivot_irr = sens.pivot_table(index='interest_rate', columns='annual_rent', values='irr', aggfunc='mean')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pivot_npv, annot=True, fmt='.0f', cmap='RdYlGn', ax=axes[0])
axes[0].set_title('NPV Sensitivity: Interest Rate vs Purchase Price')

sns.heatmap(pivot_irr, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[1])
axes[1].set_title('IRR Sensitivity: Interest Rate vs Annual Rent')

fig.tight_layout()
fig.savefig('../results/figures/sensitivity_heatmaps.png', bbox_inches='tight')
plt.show()

In [ ]:
scenario_df = pd.DataFrame([
    {'scenario': 'Bear', 'price_mu': 0.00, 'rent_mu': 0.01, 'rate': 0.068},
    {'scenario': 'Base', 'price_mu': 0.03, 'rent_mu': 0.025, 'rate': 0.056},
    {'scenario': 'Bull', 'price_mu': 0.055, 'rent_mu': 0.04, 'rate': 0.048},
])

scenario_out = []
for _, r in scenario_df.iterrows():
    a = InvestmentAssumptions(
        annual_price_growth_mu=r['price_mu'],
        rent_growth_mu=r['rent_mu'],
        annual_interest_rate=r['rate']
    )
    out = monte_carlo_investment(a, n_sims=6000, seed=101)
    scenario_out.append({
        'Scenario': r['scenario'],
        'Median IRR': out['irr'].median(),
        'Median NPV': out['npv'].median(),
        'P(NPV<0)': (out['npv'] < 0).mean(),
    })

scenario_results = pd.DataFrame(scenario_out)
scenario_results

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
tmp = scenario_results.melt(id_vars='Scenario', value_vars=['Median IRR', 'P(NPV<0)'], var_name='Metric')
sns.barplot(data=tmp, x='Scenario', y='value', hue='Metric', ax=ax)
ax.set_title('Scenario Comparison: Return vs Downside Probability')
ax.set_ylabel('Value')
fig.tight_layout()
fig.savefig('../results/figures/scenario_comparison.png', bbox_inches='tight')
plt.show()

## 7) Interpretation

- The selected ML model captures non-linear interactions that are economically intuitive (size, neighborhood quality, and macro rates dominate).
- Monte Carlo outcomes quantify risk beyond point estimates: downside probability and tail losses are directly visible.
- Sensitivity surfaces show that financing conditions and entry valuation strongly affect investment viability.

This framework can be extended with regime-switching dynamics, geographic transfer learning, and Bayesian parameter uncertainty.